In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [19]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def scrape_udemy():
    options = uc.ChromeOptions()
    # options.add_argument('--headless') # Keep this commented to see what's happening!
    
    driver = uc.Chrome(options=options, version_main=145) # Using your version from earlier
    
    try:
        url = "https://www.udemy.com/courses/it-and-software/it-certification/"
        print("Opening Udemy... If you see a CAPTCHA, please solve it manually in the browser window.")
        driver.get(url)

        # 1. Scroll down to trigger lazy loading
        driver.execute_script("window.scrollTo(0, 1000);")
        time.sleep(3) 

        # 2. Use a more generic wait. We wait for ANY link that looks like a course title.
        wait = WebDriverWait(driver, 30)
        
        # This looks for any H3 that contains a link to a course
        titles = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'h3 > a[href*="/course/"]')))

        print(f"Found {len(titles)} potential courses!\n" + "="*50)

        for title_element in titles:
            # We find the parent container of the title to get the other info
            try:
                # Move up the DOM tree to find the whole card
                card = title_element.find_element(By.XPATH, "./ancestor::div[contains(@class, 'course-card')]")
                
                title_text = title_element.text
                
                # Try to find instructor (usually near the title)
                try:
                    instructor = card.find_element(By.CSS_SELECTOR, 'div[class*="instructor-list"]').text
                except:
                    instructor = "Unknown"

                print(f"Title: {title_text}")
                print(f"Instructor: {instructor}")
                print("-" * 30)
                
            except Exception as e:
                continue

    except Exception as e:
        print(f"An error occurred: {e}")
        # Keep the browser open so you can see if a CAPTCHA appeared
        input("Press Enter to close the browser...")
    finally:
        driver.quit()

if __name__ == "__main__":
    scrape_udemy()

Opening Udemy... If you see a CAPTCHA, please solve it manually in the browser window.
Found 20 potential courses!
Title: Ultimate AWS Certified Solutions Architect Associate 2026
Full Practice Exam | Learn Cloud Computing | Pass the AWS Certified Solutions Architect Associate Certification SAA-C03!
Instructor: Stephane Maarek | AWS Certified Cloud Practitioner,Solutions Architect,Developer
------------------------------
Title: [NEW] Ultimate AWS Certified Cloud Practitioner CLF-C02 2026
Full Practice Exam included + explanations | Learn Cloud Computing | Pass the AWS Cloud Practitioner CLF-C02 exam!
Instructor: Stephane Maarek | AWS Certified Cloud Practitioner,Solutions Architect,Developer
------------------------------
Title: CompTIA Security+ (SY0-701) Complete Course & Practice Exam
CompTIA Security+ (SY0-701) Bootcamp - Your preparation for the world's best cybersecurity certification!
Instructor: Jason Dion • 2.8 Million+ Enrollments Worldwide, Dion Training Solutions • ATO for 

In [11]:
df = pd.read_csv("courses.csv")
sns.countplot(x="category", data=df)
plt.show()

EmptyDataError: No columns to parse from file

In [ ]:
X = df[["price", "duration"]]
y = df["students"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
model = RandomForestRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [ ]:
st.title("EduPulse: Course Trend Analyzer")
df = pd.read_csv("courses.csv")

st.subheader("Course Dataset")
st.write(df.head())

st.subheader("Category Distribution")
st.bar_chart(df['category'].value_counts())